# Notebook 07: Test-Time Scaling

**Time:** 25 minutes  
**Prerequisites:** Notebook 06 complete  
**Goal:** Understand and experiment with computation strategies that improve LLM quality at inference time

> 💡 **Key insight from Week 2 lecture:** We can improve model output quality without changing model weights —
> by allocating more compute *at inference time*. This is the core idea behind O1 and O3.

## Three Techniques

1. **Chain-of-Thought (CoT)** — encourage step-by-step reasoning (no extra compute cost)
2. **Extended Thinking** — model reasons in a private "scratchpad" before answering (Path A/C)
3. **Quantization** — reduce memory to allow larger models or faster inference

In [63]:
import os, sys, json, time
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir    = os.path.join('..', 'outputs')
thinking_dir   = os.path.join(outputs_dir, 'thinking_traces')
os.makedirs(thinking_dir, exist_ok=True)

print("✅ Setup complete — ready for Notebook 07")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✅ Setup complete — ready for Notebook 07


---

## Part 1: Chain-of-Thought Prompting

CoT doesn't add compute — it *reallocates* compute from the next-token prediction step to explicit reasoning tokens. The model expresses its intermediate reasoning in the output, which forces more structured thinking.

In [6]:
print("=" * 65)
print("🧪 Experiment 1: Chain-of-Thought Comparison")
print("=" * 65)
print()

# A problem that benefits from step-by-step reasoning
problem = (
    "A dataset has 10,000 documents. After language filtering, 85% remain. "
    "After MinHash deduplication (removing 30% of remaining), how many documents "
    "are left? Then 5% of what remains contains PII and is removed. "
    "What percentage of the original dataset survives the full pipeline?"
)

styles = [
    ("Direct",      problem,                                      None),
    ("Step-by-step", problem + "\n\nSolve step by step.",         None),
    ("Few-shot CoT", 
     "Example: 100 items → filter 20% → 80 left. → remove 50% → 40 left.\n"
     "Now solve:\n" + problem,
     "You are a precise data science calculator."),
]

results_exp1 = []
for style_name, prompt, system in styles:
    start = time.time()
    resp = client.generate(
        prompt=prompt, system=system,
        max_tokens=2400, temperature=0.0   # deterministic for fair comparison
    )
    elapsed = time.time() - start

    if "error" not in resp:
        tracker.add_call(resp)
        out_tokens = resp['usage']['output_tokens']
        results_exp1.append((style_name, resp['content'], out_tokens, elapsed))
        print(f"[{style_name}] {out_tokens} output tokens in {elapsed:.1f}s")
        print(f"  {resp['content'][:200]}...")
        print()

print()
print("💡 Notice how more tokens ≠ more accuracy. The STRUCTURE of reasoning matters.")

🧪 Experiment 1: Chain-of-Thought Comparison

[Direct] 326 output tokens in 8.0s
  To find the final number of documents and the survival percentage, we can calculate the results step-by-step:

### 1. Language Filtering
*   **Initial count:** 10,000 documents
*   **Remaining (85%):*...

[Step-by-step] 410 output tokens in 11.8s
  To find the final number of documents and the survival percentage, we will process the dataset through the pipeline step by step.

### Step 1: Language Filtering
The dataset starts with 10,000 documen...

[Few-shot CoT] 162 output tokens in 5.4s
  1. **Initial Dataset:** 10,000 documents
2. **Language Filtering (85% remain):** 10,000 × 0.85 = **8,500** documents
3. **MinHash Deduplication (Remove 30%):** 8,500 × 0.70 = **5,950** documents
4. **...


💡 Notice how more tokens ≠ more accuracy. The STRUCTURE of reasoning matters.


In [7]:
import json
print(json.dumps(results_exp1, indent=2))

[
  [
    "Direct",
    "To find the final number of documents and the survival percentage, we can calculate the results step-by-step:\n\n### 1. Language Filtering\n*   **Initial count:** 10,000 documents\n*   **Remaining (85%):** $10,000 \\times 0.85 = \\mathbf{8,500}$ documents\n\n### 2. MinHash Deduplication\n*   **Removed:** 30% of the remaining documents.\n*   **Remaining (70%):** $8,500 \\times 0.70 = \\mathbf{5,950}$ documents\n\n### 3. PII Removal\n*   **Removed:** 5% of the remaining documents.\n*   **Remaining (95%):** $5,950 \\times 0.95 = \\mathbf{5,652.5}$ documents\n\n*(Note: In a real-world scenario, you would have either 5,652 or 5,653 documents depending on rounding.)*\n\n---\n\n### Final Results\n\n*   **Documents remaining:** **5,652.5** (or approx. 5,653)\n*   **Percentage of original dataset:** **56.525%**\n\n**Calculation breakdown:**\n$1.00 \\times 0.85 \\times 0.70 \\times 0.95 = 0.56525$",
    326,
    8.031733989715576
  ],
  [
    "Step-by-step",
    "To find

### 🎯 TODO 1: Design a CoT Problem for Your Domain

In [9]:
# TODO 1: Create a problem from your project domain that benefits from CoT
#         Test all three styles and compare the quality of answers.
# # Examples of good CoT problems:
#   - Multi-step calculations (data pipeline, cost estimation, scheduling)
#   - Logic puzzles with constraints
#   - Clinical reasoning: "Given symptoms A, B, C — what is the most likely diagnosis?"
#   - Code debugging: "This function fails when input is X. Why? How to fix?"

my_problem = """

Three boxes are labeled:

Box A: “Apples”
Box B: “Oranges”
Box C: “Apples and Oranges”

 All labels are incorrect.

You are allowed to pick one fruit from only one box (without looking inside).

Which box do you pick from, and how do you correctly relabel all three boxes after that one pick?

"""

print("=" * 65)
print("🎯 TODO 1: CoT on My Domain Problem")
print("=" * 65)
print()

for style_name, suffix in [("Direct", ""), ("CoT", "\n\nSolve step by step.")]:
    resp = client.generate(
        prompt=my_problem + suffix,
        max_tokens=2500, temperature=0.0
    )
    if "error" not in resp:
        tracker.add_call(resp)
        print(f"[{style_name}] {resp['usage']['output_tokens']} tokens")
        print(resp['content'])
        print()


🎯 TODO 1: CoT on My Domain Problem

[Direct] 390 tokens
To solve this, you must pick a fruit from **Box C (labeled "Apples and Oranges")**.

Here is the step-by-step logic to relabel the boxes:

### 1. The Pick
Because you know **all labels are incorrect**, Box C cannot contain both apples and oranges. It must contain *only* apples or *only* oranges.

### 2. Identify Box C
*   If you pick an **Apple** from Box C, then Box C is the **Apples** box.
*   If you pick an **Orange** from Box C, then Box C is the **Oranges** box.

### 3. Identify the remaining two boxes
Let’s assume you picked an **Apple** from Box C. You now know:
*   **Box C:** Apples
*   **Box A (Labeled "Apples"):** Must be either "Oranges" or "Both."
*   **Box B (Labeled "Oranges"):** Must be either "Apples" or "Both."

Since you have already identified the "Apples" box (Box C), Box B cannot be "Apples." Therefore, **Box B must be "Both."**

This leaves only one option for the final box: **Box A must be "Oranges."**

---


In [12]:

# TODO 1: Create a problem from your project domain that benefits from CoT
#         Test all three styles and compare the quality of answers.

my_problem = """

A 3-digit number ABC (digits A,B,C) satisfies:

The digits are all different and A<>0
The number is divisible by 9
Reversing the digits gives a number that is 36 less than the original

The sum of the first and last digits equals the middle digit:

A+C=B
What is the number ABC?
"""

print("=" * 65)
print("🎯 TODO 1: CoT on My Domain Problem")
print("=" * 65)
print()

for style_name, suffix in [("Direct", ""), ("CoT", "\n\nSolve step by step.")]:
    resp = client.generate(
        prompt=my_problem + suffix,
        max_tokens=4000, temperature=0.0
    )
    if "error" not in resp:
        tracker.add_call(resp)
        print(f"[{style_name}] {resp['usage']['output_tokens']} tokens")
        print(resp['content'])
        print()

🎯 TODO 1: CoT on My Domain Problem

[Direct] 160 tokens
To find the 3-digit number **ABC**, we will use the conditions provided:

1.  **The digits are all different and $A \neq 0$.**
2.  **The number is divisible by 9.**
    *   A number is divisible by 9 if the sum of its digits is a multiple of 9.
    *   $A + B + C = 9k$ (where $k$ is an integer).
3.  **The sum of the first and last digits equals the middle digit: $A + C = B$.**
    *   Substitute $B$ into the sum of digits equation: $A + (A + C) + C = 2(A + C) = 2

[CoT] 156 tokens
$.
    We are given $A+C=B$.
    So $2B$ must be a multiple of 9.
    $B$ must be 9 (since $B$ is a digit and $A \neq 0$).
    If $B=9$, then $A+C=9$.
    If $A+C=9$, the possible values for $A-C$ are:
    $A=8, C=1 \implies A-C=7 \implies 99 \times 7 = 693$
    $A=7, C=2 \implies A-C=5 \implies 99 \times 5 = 495$
    



In [13]:

todo1_reflection = """


- Which style produced the more accurate answer? How do you know?
 They both produced the same answer, which is correct. However, the CoT style provided a detailed reasoning process that clearly showed how it arrived at the answer,
  while the direct style just gave the final answer with simple explanation. Maybe need more complex problem to see the difference.
- Did CoT use more tokens? Was that extra cost worth it?
Sometimes CoT used more tokens, but not always. In this case, the CoT style used more tokens due to the detailed reasoning steps. 
    Whether it's worth it depends on the context; for complex problems where understanding the reasoning is crucial, 
    it can be worth it. For simpler problems, the direct style may suffice.
- For what types of problems does CoT help the most in your domain?
    CoT is especially helpful for problems that require multi-step reasoning, such as data pipeline calculations, cost estimations, scheduling, logic puzzles with constraints, clinical reasoning, and code debugging. 
    In these cases, the step-by-step approach allows for a clearer understanding of the problem-solving process and can lead to more accurate and insightful answers.
"""
print(todo1_reflection)




- Which style produced the more accurate answer? How do you know?
 They both produced the same answer, which is correct. However, the CoT style provided a detailed reasoning process that clearly showed how it arrived at the answer,
  while the direct style just gave the final answer with simple explanation. Maybe need more complex problem to see the difference.
- Did CoT use more tokens? Was that extra cost worth it?
Sometimes CoT used more tokens, but not always. In this case, the CoT style used more tokens due to the detailed reasoning steps. 
    Whether it's worth it depends on the context; for complex problems where understanding the reasoning is crucial, 
    it can be worth it. For simpler problems, the direct style may suffice.
- For what types of problems does CoT help the most in your domain?
    CoT is especially helpful for problems that require multi-step reasoning, such as data pipeline calculations, cost estimations, scheduling, logic puzzles with constraints, clinica

---

## Part 2: Extended Thinking (Path A/C — Claude API)

Claude's **extended thinking** lets the model reason in a private scratchpad before producing its final answer. This is similar to OpenAI's o1/o3 reasoning tokens.

Unlike CoT (where reasoning is in the output), extended thinking:
- Reasoning is hidden from the user by default (but accessible via the API)
- Budget controls how many tokens to spend on reasoning
- Particularly effective for math, logic, multi-step problems

In [38]:
print("=" * 65)
print("🧪 Experiment 2: Extended Thinking")
print("=" * 65)
print()

logic_puzzle = """
Five ML engineers (Alice, Bob, Carol, Dave, Eve) are assigned to one of five tasks:
data collection, cleaning, tokenization, pretraining, and evaluation.

Constraints:
1. Alice cannot do pretraining (insufficient GPU budget)
2. Bob is doing either data collection or cleaning
3. Carol is doing tokenization or evaluation
4. Dave and Eve are not doing adjacent pipeline stages
   (collection→cleaning→tokenization→pretraining→evaluation)
5. Eve is not doing data collection

Who does what? Show all your reasoning.
"""

if config.PATH in ["A", "C"]:
    print("Running extended thinking (budget_tokens=2000)...")
    start = time.time()
    
    resp_thinking = client.generate_with_thinking(
        prompt=logic_puzzle,
        budget_tokens=2000,
        max_tokens=4096
    )
    elapsed = time.time() - start

    if "error" not in resp_thinking:
        tracker.add_call(resp_thinking)

        print(f"✅ Response in {elapsed:.1f}s")
        print(f"   Tokens: {resp_thinking['usage']['input_tokens']}in / {resp_thinking['usage']['output_tokens']}out")
        print()
        print("--- THINKING (internal reasoning) ---")
        thinking = resp_thinking.get('thinking', '')
        print(thinking[:600] + ('...' if len(thinking) > 600 else ''))
        print()
        print("--- FINAL ANSWER ---")
        print(resp_thinking['content'])

        # Save thinking trace
        trace_path = os.path.join(thinking_dir, 'logic_puzzle_thinking.json')
        with open(trace_path, 'w') as f:
            json.dump({
                "prompt": logic_puzzle,
                "thinking": resp_thinking.get('thinking', ''),
                "answer": resp_thinking['content'],
                "budget_tokens": 2000,
                "usage": resp_thinking['usage'],
            }, f, indent=2)
        print(f"\n✅ Thinking trace saved: {trace_path}")
    else:
        print(f"❌ Error: {resp_thinking['error']}")

else:
    print("Path B (Ollama) — using multi-turn scratchpad simulation instead")
    print()
    # Simulate extended thinking with explicit scratchpad
    scratchpad_prompt = (
        "Before answering, work through this step by step in a <thinking> block. "
        "Then give your final answer after </thinking>.\n\n" + logic_puzzle
    )
    resp_scratchpad = client.generate(prompt=scratchpad_prompt, max_tokens=800, temperature=0.0)
    if "error" not in resp_scratchpad:
        tracker.add_call(resp_scratchpad)
        print(format_response(resp_scratchpad, verbose=True))

🧪 Experiment 2: Extended Thinking

Path B (Ollama) — using multi-turn scratchpad simulation instead

Model: gemini-3-flash-preview
Tokens: 149 in, 796 out
Stop reason: FinishReason.MAX_TOKENS
To determine the assignment of tasks for the five ML engineers, we analyze the constraints step-by-step:

<thinking>
I began by mapping the pipeline stages to a numerical sequence (1-5) and listing the constraints for each engineer. By testing the limited possibilities for Bob (Collection or Cleaning) and Carol (Tokenization or Evaluation) against the adjacency constraint for Dave and Eve, I identified that Bob must be assigned to Cleaning and Carol to Evaluation. This configuration allows Dave and Eve to occupy non-adjacent slots (Collection and Pretraining), leaving the remaining Tokenization task for Alice.
</thinking>

Based on the constraints provided, here is the step-by-step reasoning to find the unique solution:

1.  **Identify the Pipeline Order:**
    1. Data Collection
    2. Cleaning
 

In [64]:
print("=" * 65)
print("🧪 Experiment 2: Extended Thinking")
print("=" * 65)
print()

logic_puzzle = """
Five ML engineers (Alice, Bob, Carol, Dave, Eve) are assigned to one of five tasks:
data collection, cleaning, tokenization, pretraining, and evaluation.

Constraints:
1. Alice cannot do pretraining (insufficient GPU budget)
2. Bob is doing either data collection or cleaning
3. Carol is doing tokenization or evaluation
4. Dave and Eve are not doing adjacent pipeline stages
   (collection→cleaning→tokenization→pretraining→evaluation)
5. Eve is not doing data collection

Who does what? Show all your reasoning.
"""

if config.PATH in ["A", "C"]:
    print("Running extended thinking (budget_tokens=2000)...")
    start = time.time()
    
    resp_thinking = client.generate_with_thinking(
        prompt=logic_puzzle,
        budget_tokens=2000,
        max_tokens=4096
    )
    elapsed = time.time() - start

    if "error" not in resp_thinking:
        tracker.add_call(resp_thinking)

        print(f"✅ Response in {elapsed:.1f}s")
        print(f"   Tokens: {resp_thinking['usage']['input_tokens']}in / {resp_thinking['usage']['output_tokens']}out")
        print()
        print("--- THINKING (internal reasoning) ---")
        thinking = resp_thinking.get('thinking', '')
        print(thinking[:600] + ('...' if len(thinking) > 600 else ''))
        print()
        print("--- FINAL ANSWER ---")
        print(resp_thinking['content'])

        # Save thinking trace
        trace_path = os.path.join(thinking_dir, 'logic_puzzle_thinking.json')
        with open(trace_path, 'w') as f:
            json.dump({
                "prompt": logic_puzzle,
                "thinking": resp_thinking.get('thinking', ''),
                "answer": resp_thinking['content'],
                "budget_tokens": 2000,
                "usage": resp_thinking['usage'],
            }, f, indent=2)
        print(f"\n✅ Thinking trace saved: {trace_path}")
    else:
        print(f"❌ Error: {resp_thinking['error']}")


🧪 Experiment 2: Extended Thinking

Running extended thinking (budget_tokens=2000)...
✅ Response in 58.5s
   Tokens: 162in / 4096out

--- THINKING (internal reasoning) ---
Let me work through this systematically.

Tasks: data collection (1), cleaning (2), tokenization (3), pretraining (4), evaluation (5)
People: Alice, Bob, Carol, Dave, Eve

Constraints:
1. Alice ≠ pretraining
2. Bob = data collection OR cleaning
3. Carol = tokenization OR evaluation
4. Dave and Eve not adjacent (stages differ by more than 1 in the pipeline order)
5. Eve ≠ data collection

From constraint 2: Bob is in {collection, cleaning}


From constraint 3: Carol is in {tokenization, evaluation}
From constraint 5: Eve ≠ collection

Since Bob handles either collection or cleaning, and Carol h...

--- FINAL ANSWER ---


✅ Thinking trace saved: ../outputs/thinking_traces/logic_puzzle_thinking.json


In [61]:
import importlib
import src.config
importlib.reload(src.config)
import src.config  # re-import!
import src.llm_client
importlib.reload(src.llm_client)
from src.llm_client import LLMClient  # re-import!

%load_ext autoreload
%autoreload 2
print(src.config.PATH)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
A


### 🎯 TODO 2: Your Project Domain + Extended Thinking Budget Comparison

In [68]:
# TODO 2: Run extended thinking on a problem from your project domain
#         Compare budget_tokens=500 vs budget_tokens=3000
# Good candidates:
#   - A complex design decision: "Should I use RAG or fine-tuning for [use case]?"
#   - A tricky data analysis question in your domain
#   - An ethical dilemma about AI deployment in your field
#   - A multi-constraint optimization problem

my_thinking_problem = """
Is it possible to complete revenue tax declaration by uploading documents and providing additional information, and then complete the report by LLM?
If possible, what are the steps to complete the tax declaration? What are the potential challenges and limitations of using LLM for this task?
web version or local version? What are the differences between these two versions in terms of performance, security, and usability?
How to ensure the accuracy and reliability of the tax declaration generated by LLM? What are the potential risks and mitigation strategies for using LLM in this context?

 
"""

print("=" * 65)
print("🎯 TODO 2: Extended Thinking Budget Comparison")
print("=" * 65)
print()

budgets = [500, 3000]
budget_results = []

for budget in budgets:
    print(f"\n--- Budget: {budget} tokens ---")
    if config.PATH in ["A", "C"]:
        
        resp = client.generate_with_thinking(
            prompt=my_thinking_problem,
            budget_tokens=budget,
            max_tokens=budget + 2048
        )
    else:
        # Path B fallback
        
        resp = client.generate(
            prompt=f"Think step by step with {budget} words of reasoning.\n{my_thinking_problem}",
            max_tokens=budget
        )

    if "error" not in resp:
        tracker.add_call(resp)
        answer = resp['content']
        thinking_len = len(resp.get('thinking', ''))
        budget_results.append((budget, answer, thinking_len))

        print(f"  Thinking length: {thinking_len} chars")
        print(f"  Answer preview:  {answer[:300]}...")

        # Save trace
        trace_path = os.path.join(thinking_dir, f'project_problem_budget{budget}.json')
        with open(trace_path, 'w') as f:
            json.dump({
                "prompt": my_thinking_problem,
                "budget_tokens": budget,
                "thinking": resp.get('thinking', ''),
                "answer": answer,
                "usage": resp['usage'],
            }, f, indent=2)
        print(f"  ✅ Saved: {trace_path}")


🎯 TODO 2: Extended Thinking Budget Comparison


--- Budget: 500 tokens ---

--- Budget: 3000 tokens ---
  Thinking length: 167 chars
  Answer preview:  # Using LLM for Tax Declaration Completion

## 1. Is It Possible?

```
YES - Technically feasible, but with significant caveats
┌─────────────────────────────────────────────────────┐
│  Current Capability Level:                          │
│  ✅ Document parsing and extraction                 │
│  ✅ ...
  ✅ Saved: ../outputs/thinking_traces/project_problem_budget3000.json


In [69]:

todo2_reflection = """
- How did the answer quality change between budget=500 and budget=3000?
 Since the input problem is quite complex and requires detailed reasoning, there is no output, since the input has already arrived 500 tokens. 
 With a budget of 3000 tokens, the model provided a very high-level answer, however only a brief summary because of the limited token budget.

- Was the quality improvement worth the extra cost? (More output tokens = more cost)
    The quality improvement from 500 to 3000 tokens was significant, as the model was able to provide a much more detailed and comprehensive answer with the larger budget. 
    However, whether it was worth the extra cost depends on the specific use case and requirements. For critical tasks where accuracy and depth of reasoning are essential, the extra cost may be justified. 
    For simpler tasks or when budget constraints are tight, the smaller budget might be sufficient.

- For your production use case, what budget_tokens setting would you choose?
 I would choose a budget_tokens setting of around 2000-4000 for my production use case, as it provides a good balance between answer quality
  and cost.

- When would you use extended thinking vs. standard CoT?
    Extended thinking is particularly beneficial for complex problems that require deep reasoning and multi-step solutions, as it allows the model to generate intermediate thoughts and refine its answer iteratively. 
    Standard CoT may be sufficient for simpler problems or when a quick answer is needed without the need for detailed reasoning. 
    The choice between the two would depend on the complexity of the problem and the importance of having a well-reasoned answer.

"""
print()
print(todo2_reflection)



- How did the answer quality change between budget=500 and budget=3000?
 Since the input problem is quite complex and requires detailed reasoning, there is no output, since the input has already arrived 500 tokens. 
 With a budget of 3000 tokens, the model provided a very high-level answer, however only a brief summary because of the limited token budget.

- Was the quality improvement worth the extra cost? (More output tokens = more cost)
    The quality improvement from 500 to 3000 tokens was significant, as the model was able to provide a much more detailed and comprehensive answer with the larger budget. 
    However, whether it was worth the extra cost depends on the specific use case and requirements. For critical tasks where accuracy and depth of reasoning are essential, the extra cost may be justified. 
    For simpler tasks or when budget constraints are tight, the smaller budget might be sufficient.

- For your production use case, what budget_tokens setting would you choos

---

## Part 3: Quantization — More Model for Less Memory

In [70]:
print("=" * 65)
print("🧪 Experiment 3: Float32 vs Float16 Memory & Speed")
print("=" * 65)
print()

try:
    import torch
    
    # Model size: realistic embedding layer (vocab=50k, dim=512)
    vocab_size = 50_000
    d_model    = 512
    
    # Float32 (standard)
    embedding_f32 = torch.nn.Embedding(vocab_size, d_model)  # 4 bytes per param
    params_f32 = embedding_f32.weight.numel()
    bytes_f32  = params_f32 * 4
    
    # Float16 (half precision)
    embedding_f16 = embedding_f32.half()                      # 2 bytes per param
    bytes_f16  = params_f32 * 2

    print(f"Embedding layer: vocab={vocab_size}, dim={d_model}")
    print(f"  Parameters: {params_f32:,}")
    print()
    print(f"  float32 size: {bytes_f32 / 1024**2:.1f} MB")
    print(f"  float16 size: {bytes_f16 / 1024**2:.1f} MB")
    print(f"  Memory saved: {(1 - bytes_f16/bytes_f32)*100:.0f}%")
    print()

    # Speed benchmark
    x = torch.randint(0, vocab_size, (32, 512))  # batch=32, seq_len=512

    # Warm up
    for _ in range(3):
        _ = embedding_f32(x)

    # Float32 timing
    start = time.time()
    for _ in range(100):
        _ = embedding_f32(x)
    t32 = time.time() - start

    # Float16 timing
    start = time.time()
    for _ in range(100):
        _ = embedding_f16(x)
    t16 = time.time() - start

    speedup = t32 / t16
    print(f"  float32 time (100 batches): {t32:.3f}s")
    print(f"  float16 time (100 batches): {t16:.3f}s")
    print(f"  Speedup: {speedup:.1f}x")
    print()

    # Scale to a real model
    print("Scaling to full model sizes:")
    print()
    for model_name, params_B in [("LLaMA 7B", 7), ("LLaMA 13B", 13), ("LLaMA 70B", 70)]:
        f32_gb = params_B * 4
        f16_gb = params_B * 2
        int8_gb = params_B * 1
        print(f"  {model_name:<12}  float32: {f32_gb}GB  float16: {f16_gb}GB  int8: {int8_gb}GB")

    print()
    print("💡 A consumer A100 (80GB) can run LLaMA 70B in float16 but NOT float32.")
    print("   QLoRA (4-bit quantization) fits 70B in a single 40GB A100!")

except ImportError:
    print("⚠️  PyTorch not installed — skipping experiment")


🧪 Experiment 3: Float32 vs Float16 Memory & Speed

Embedding layer: vocab=50000, dim=512
  Parameters: 25,600,000

  float32 size: 97.7 MB
  float16 size: 48.8 MB
  Memory saved: 50%

  float32 time (100 batches): 0.117s
  float16 time (100 batches): 0.159s
  Speedup: 0.7x

Scaling to full model sizes:

  LLaMA 7B      float32: 28GB  float16: 14GB  int8: 7GB
  LLaMA 13B     float32: 52GB  float16: 26GB  int8: 13GB
  LLaMA 70B     float32: 280GB  float16: 140GB  int8: 70GB

💡 A consumer A100 (80GB) can run LLaMA 70B in float16 but NOT float32.
   QLoRA (4-bit quantization) fits 70B in a single 40GB A100!


In [72]:

todo3_reflection = """

- What speedup did you observe for float16 vs float32?
    In the benchmark, we observed a speedup when using float32 compared to float16. 
    It is out of expectation, since float16 is generally expected to be faster than float32 due to reduced memory bandwidth and better GPU utilization.
    Since I used CPU, so there is no benefit from float16, and the overhead of using float16 on CPU may actually make it slower than float32.
    The reason are following:
    1. CPU does not have native support for float16, so it has to emulate float16 operations using float32, which can introduce overhead and reduce performance.
    2. The reduced precision of float16 can lead to increased numerical instability and more frequent overflows/underflows, which can further degrade performance on CPU. 
    3. Many operations internally convert:float16 → float32 → compute → float16


- For qwen3.5:27b (your local model), approximately how much GPU memory does it need
    The qwen3.5:27b model, when loaded in float16 precision, would require approximately 27 billion parameters * 2 bytes/parameter = 54 GB of GPU memory. 
    This is a rough estimate and the actual memory usage may be higher due to additional overhead from the model architecture, activations, and other factors. 
    However, it should fit on a consumer GPU with 24GB of VRAM if we use techniques like model parallelism or offloading parts of the model to CPU memory.

  in float16? Would it fit on a consumer GPU (24GB)?
    The qwen3.5:27b model in float16 would require approximately 54 GB of GPU memory, which exceeds the 24 GB available on a consumer GPU. 
    Therefore, it would not fit on a single consumer GPU without using techniques like model parallelism, offloading, or quantization to reduce memory usage.
- What's the trade-off between int8 quantization vs float16?
    The trade-off between int8 quantization and float16 precision involves a balance between memory efficiency, computational speed, and model accuracy. 
    Int8 quantization reduces the model size by representing weights and activations with 8 bits instead of 16 bits (float16) or 32 bits (float32), which can lead to significant memory savings and faster inference times. 
    However, this reduction in precision can also lead to a loss in model accuracy, especially for models that are sensitive to quantization. 
    Float16 offers a middle ground, providing reduced memory usage compared to float32 while maintaining better accuracy than int8 quantization.

- How does quantization relate to test-time scaling? (Hint: same GPU budget = ?)
    Quantization allows us to fit larger models or use more of the model's capacity within the same GPU memory budget. 
    For example, if a model in float16 requires 54 GB of memory, quantizing it to int8 could reduce that requirement to around 27 GB, allowing it to fit on a 24 GB GPU with some room for activations and overhead. 
    This means that with quantization, we can potentially use a more powerful model or allocate more tokens for reasoning within the same GPU budget, which can enhance test-time scaling and improve performance on complex tasks. 

"""
print()
print(todo3_reflection)




- What speedup did you observe for float16 vs float32?
    In the benchmark, we observed a speedup when using float32 compared to float16. 
    It is out of expectation, since float16 is generally expected to be faster than float32 due to reduced memory bandwidth and better GPU utilization.
    Since I used CPU, so there is no benefit from float16, and the overhead of using float16 on CPU may actually make it slower than float32.
    The reason are following:
    1. CPU does not have native support for float16, so it has to emulate float16 operations using float32, which can introduce overhead and reduce performance.
    2. The reduced precision of float16 can lead to increased numerical instability and more frequent overflows/underflows, which can further degrade performance on CPU. 
    3. Many operations internally convert:float16 → float32 → compute → float16


- For qwen3.5:27b (your local model), approximately how much GPU memory does it need
    The qwen3.5:27b model, when lo

## Summary & Reflection

In [73]:
full_reflection = f"""
### Chain-of-Thought Experiment (TODO 1)

My problem domain: {my_problem[:100].strip() if 'my_problem' in dir() else '[not set]'}

{todo1_reflection.strip()}

---

### Extended Thinking Budget Comparison (TODO 2)

Budget results: {[(b, len(a)) for b, a, _ in budget_results] if 'budget_results' in dir() else 'N/A'}

{todo2_reflection.strip()}

---

### Quantization Speedup (TODO 3 — Experiment 3)

{todo3_reflection.strip()}
"""

reflection_file = append_to_reflection(
    notebook="07",
    section_title="Test-Time Scaling Experiments",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)
print(f"✅ Reflection saved: {reflection_file}")
print(f"✅ Thinking traces saved to: {thinking_dir}")
print()
tracker.report()

✅ Reflection saved: ../outputs/homework_reflection.md
✅ Thinking traces saved to: ../outputs/thinking_traces

💰 API COST REPORT
Total API calls:     2
Total input tokens:  319
Total output tokens: 7,607
Total cost:          $0.1151

Last 2 calls:
  1. [14:00:03] sonnet — 162in/4096out — $0.0619
  2. [15:33:35] sonnet — 157in/3511out — $0.0531


## ✅ Notebook 07 Complete!

**What you accomplished:**
- ✅ Compared direct vs CoT vs few-shot CoT on a multi-step problem
- ✅ Used Claude's extended thinking API and inspected the thinking trace
- ✅ Measured float32 vs float16 memory and speed difference
- ✅ Saved thinking traces to `outputs/thinking_traces/`

**Key concepts:**
- CoT = more reasoning tokens at no extra per-token cost, better structured answers
- Extended thinking = private reasoning scratchpad, more powerful than CoT for hard problems
- Quantization = same accuracy, 2–4× less memory, enables running larger models locally
- Test-time scaling: allocate more compute at inference → better output without retraining

**Next:** Open **Notebook 08: Project Integration** 🚀